# RFM Visualization – Roll D

In [21]:
import pandas as pd
import plotly.express as px
from supabase import create_client

In [22]:
import os
from dotenv import load_dotenv

load_dotenv('/Users/marianne/DACA/.env')

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')

print('URL olemas:', bool(SUPABASE_URL))
print('Võti olemas:', bool(SUPABASE_KEY))

URL olemas: True
Võti olemas: True


In [9]:
from dotenv import load_dotenv

load_dotenv('/Users/marianne/DACA/.env', override=True)

api_key = os.getenv('SUPABASE_KEY', '')
print('Võtme algus:', api_key[:15])
print('Võtme pikkus:', len(api_key))

Võtme algus: sb_publishable_
Võtme pikkus: 46


In [11]:
import pandas as pd
import plotly.express as px
import os
from dotenv import load_dotenv
from supabase import create_client

In [8]:
import os

In [16]:
supabase = create_client(
    os.getenv('SUPABASE_URL'),
    os.getenv('SUPABASE_KEY')
)

print('Uus Supabase ühendus loodud!')

Uus Supabase ühendus loodud!


## 1. RFM visualiseerimise ettevalmistamine

Selles osas loome kolm interaktiivset Plotly diagrammi:
- RFM-segmentide jaotus
- Klientide hajuvusdiagramm (Recency vs Monetary Value)
- Top 10 VIP-klienti kulutuste järgi

RFM andmed lisame pärast seda, kui Roll C on RFM tabeli valmis saanud.

## 2. Diagrammide ühine kujundus

Kõik diagrammid kasutavad ühtset kujundust:
- Diagrammide pealkirjad on eesti keeles.
- Telgede nimetused on eesti keeles.
- Diagrammid on interaktiivsed.
- Segmendid on eristatud värvidega.

In [12]:
import plotly.io as pio

# UrbanStyle diagrammide ühine kujundusmall
urbanstyle_template = pio.templates['plotly_white']

urbanstyle_template.layout.update(
    font=dict(
        family='Arial',
        size=13,
        color='#2C3E50'
    ),
    title=dict(
        font=dict(size=20)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=60, r=40, t=70, b=60)
)

pio.templates['urbanstyle'] = urbanstyle_template

In [13]:
# RFM-segmentide värvid
segment_colors = {
    'VIP Champions': '#0D9488',
    'Loyal Customers': '#2563EB',
    'At Risk': '#F59E0B',
    'Lost Customers': '#DC2626',
    'New Customers': '#8B5CF6',
    'Potential Loyalists': '#06B6D4'
}

In [14]:
# Ühine diagrammide kujunduse funktsioon

def kujunda_diagramm(fig, pealkiri):
    fig.update_layout(
        template='urbanstyle',
        title=pealkiri,
        hovermode='closest',
        legend_title_text='Kliendisegment',
        height=550
    )

    return fig

## 3. RFM-tulemuste äriline tõlgendus

### Peamised leiud

RFM-analüüsi põhjal hindame klientide ostuaktiivsust, ostusagedust ja kogukulutusi. Diagrammid aitavad tuvastada kõige väärtuslikumad kliendisegmendid ning kliendid, kelle aktiivsus on vähenenud.

### Ärilised soovitused

- **VIP-programm:** pakkuda kõige väärtuslikumatele klientidele personaalseid soodustusi, lojaalsusboonuseid ja varajast ligipääsu uutele toodetele.
- **Win-back kampaania:** võtta ühendust At-Risk klientidega ning pakkuda neile spetsiaalseid tagasikutsumise pakkumisi.
- **Uute klientide kaasamine:** julgustada uusi kliente kordusoste tegema personaalse kommunikatsiooni ja ostujärgsete pakkumiste abil.

### Järeldused

Tegelikud järeldused ja soovituste täpsem põhjendus lisatakse pärast RFM-tabeli ja diagrammide valmimist.

## 4. Diagrammide kirjeldused

### Diagramm 1 – RFM-segmentide jaotus
Näitab, kui palju kliente kuulub igasse RFM-segmenti.

- X-telg: Kliendisegment
- Y-telg: Klientide arv

### Diagramm 2 – Klientide hajuvusdiagramm
Näitab klientide viimase ostu kauguse (Recency) ja kogukulutuste (Monetary Value) vahelist seost.

- X-telg: Päevi viimasest ostust
- Y-telg: Kogukulutus (€)
- Värv: Kliendisegment
- Punkti suurus: Ostude arv

### Diagramm 3 – Top 10 VIP-klienti
Näitab kümmet kõige rohkem kulutanud VIP-klienti.

- X-telg: Kliendi ID
- Y-telg: Kogukulutus (€)

In [15]:
# RFM-tabeli kontrollfunktsioon

def kontrolli_rfm_tabel(rfm):
    vajalikud_veerud = [
    'recency_days',
    'frequency',
    'monetary_value',
    'Segment'
]

    puuduvad = [
        veerg for veerg in vajalikud_veerud
        if veerg not in rfm.columns
    ]

    if puuduvad:
        print('Puuduvad veerud:', puuduvad)
    else:
        print('Kõik vajalikud veerud on olemas.')

    print('Tabeli mõõtmed:', rfm.shape)

In [16]:
# Top 10 VIP-kliendi diagramm

def loo_vip_diagramm(rfm):
    vip = rfm[rfm['Segment'] == 'VIP Champions'].nlargest(
        10, 'Monetary'
    )

    fig = px.bar(
        vip,
        x='customer_id',
        y='Monetary',
        title='Top 10 VIP-klienti kogukulutuste järgi',
        labels={
            'customer_id': 'Kliendi ID',
            'Monetary': 'Kogukulutus (€)'
        },
        color_discrete_sequence=['#0D9488']
    )

    fig.update_layout(
        template='urbanstyle',
        xaxis_title='Kliendi ID',
        yaxis_title='Kogukulutus (€)'
    )

    return fig

In [17]:
# RFM-segmentide jaotuse diagramm

def loo_segmentide_jaotus(rfm):
    segmentide_arv = (
        rfm['Segment']
        .value_counts()
        .reset_index()
    )

    segmentide_arv.columns = ['Segment', 'Klientide arv']

    fig = px.bar(
        segmentide_arv,
        x='Segment',
        y='Klientide arv',
        color='Segment',
        color_discrete_map=segment_colors,
        title='Klientide jaotus RFM-segmentide järgi',
        labels={
            'Segment': 'Kliendisegment',
            'Klientide arv': 'Klientide arv'
        }
    )

    fig = kujunda_diagramm(
        fig,
        'Klientide jaotus RFM-segmentide järgi'
    )

    return fig

In [18]:
# Klientide hajuvusdiagramm: Recency vs Monetary Value

def loo_hajuvusdiagramm(rfm):
    fig = px.scatter(
        rfm,
        x='recency_days',
        y='monetary_value',
        color='Segment',
        size='frequency',
        color_discrete_map=segment_colors,
        hover_data=['frequency'],
        title='Klientide aktiivsus ja kulutused',
        labels={
            'recency_days': 'Päevi viimasest ostust',
            'monetary_value': 'Kogukulutus (€)',
            'frequency': 'Ostude arv',
            'Segment': 'Kliendisegment'
        }
    )

    fig = kujunda_diagramm(
        fig,
        'Klientide aktiivsus ja kulutused'
    )

    return fig

In [19]:
# Klientide hajuvusdiagramm

def loo_hajuvusdiagramm(rfm):
    fig = px.scatter(
        rfm,
        x='recency_days',
        y='monetary_value',
        color='Segment',
        size='frequency',
        hover_data=['customer_id'],
        title='Klientide ostuaktiivsus ja kogukulutused',
        labels={
            'recency_days': 'Päevi viimasest ostust',
            'monetary_value': 'Kogukulutus (€)',
            'frequency': 'Ostude arv',
            'Segment': 'Kliendisegment'
        },
        color_discrete_map=segment_colors
    )

    fig.update_layout(template='urbanstyle')

    return fig

In [20]:
# Top 10 VIP-kliendi kulutuste diagramm

def loo_top10_vip(rfm):
    vip_kliendid = (
        rfm[rfm['Segment'] == 'VIP Champions']
        .nlargest(10, 'monetary_value')
        .copy()
    )

    vip_kliendid['Klient'] = vip_kliendid.index.astype(str)

    fig = px.bar(
        vip_kliendid,
        x='Klient',
        y='monetary_value',
        color='Segment',
        color_discrete_map=segment_colors,
        title='Top 10 VIP-kliendid kulutuste järgi',
        labels={
            'Klient': 'Klient',
            'monetary_value': 'Kogukulutus (€)',
            'Segment': 'Kliendisegment'
        }
    )

    fig = kujunda_diagramm(
        fig,
        'Top 10 VIP-kliendid kulutuste järgi'
    )

    return fig

## 3. RFM-tulemuste äritõlgendus

RFM-analüüsi abil jaotatakse kliendid ostukäitumise põhjal segmentidesse. 
Analüüs aitab hinnata klientide aktiivsust, ostusagedust ja kogukulutusi.

### Peamised leiud

- VIP Champions – kõige väärtuslikumad ja aktiivsemad kliendid.
- Loyal Customers – lojaalsed kliendid, kelle ostukäitumist tasub toetada.
- At Risk – kliendid, kelle aktiivsus on vähenenud ja kes võivad ettevõttest eemalduda.
- Lost Customers – kliendid, kes pole pikemat aega oste teinud.
- New Customers – hiljuti lisandunud kliendid.
- Potential Loyalists – kliendid, kellest võivad kujuneda lojaalsed püsikliendid.

*Lõplikud järeldused täiendatakse pärast RFM-tabeli saamist.*

## 4. Turundussoovitused

### VIP-programm
VIP Champions klientidele tuleks pakkuda eksklusiivseid soodustusi, varajast ligipääsu uutele toodetele ja personaalseid pakkumisi. Eesmärk on säilitada nende lojaalsus ja suur ostuaktiivsus.

### Win-back kampaania
At Risk ja Lost Customers segmentidele tuleks saata personaalseid tagasikutsuvaid pakkumisi, näiteks erisoodustusi või tasuta transporti. Eesmärk on motiveerida kliente uuesti ostma.

### Uute klientide kaasamine
New Customers ja Potential Loyalists segmentidele võiks pakkuda tervituspakkumisi, ostusoovitusi ja lojaalsusprogrammi liitumise võimalust. Eesmärk on kujundada uutest klientidest püsikliendid.

*Soovitusi täpsustatakse pärast RFM-tulemuste analüüsimist.*

In [37]:
# Loeme kõik customers tabeli andmed Supabase'ist
all_customers = []
start = 0
batch_size = 1000

while True:
    response = (
        supabase.table('customers')
        .select('*')
        .range(start, start + batch_size - 1)
        .execute()
    )

    batch = response.data
    all_customers.extend(batch)

    if len(batch) < batch_size:
        break

    start += batch_size

# Loome DataFrame'i
customers_df = pd.DataFrame(all_customers)

# Leiame kuupäevaveerud ja teisendame need kuupäevadeks
date_columns = [
    col for col in customers_df.columns
    if 'date' in col.lower()
]

for col in date_columns:
    customers_df[col] = pd.to_datetime(
        customers_df[col],
        format='mixed',
        errors='coerce'
    )

# Kuvame kogu tabeli
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Customers ridu kokku:', len(customers_df))
print('Kuupäevaveerud:', date_columns)

display(customers_df)

Customers ridu kokku: 3150
Kuupäevaveerud: ['registration_date']


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992
...,...,...,...,...,...,...,...,...,...
3145,5146,tõnu,Roots,tonu.roots@telia.ee,+372 5875 9834,Tartu,2021-06-28,silver,2003
3146,5147,katrin,Paas,katrin.paas@mail.ee,+372 5634 3599,Tallinn,2022-11-23,silver,2003
3147,5148,Nele,Rebane,nele.rebane@yahoo.com,+372 5149 6896,Tallinn,2021-10-27,NaN,1983
3148,5149,KÜLLI,Sild,kulli.sild@mail.ee,+372 5439 3553,Tartu,2023-08-10,bronze,1994
